In [ ]:
# basic notebook for plotting logos from annotated seqlets for highlighting specific examples #

In [14]:
# import packages
import pandas as pd
import numpy as np
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
import torch
from collections import Counter
# tangermeme packages
from tangermeme.seqlet import recursive_seqlets
from tangermeme.plot import plot_logo
from tangermeme.annotate import annotate_seqlets
from tangermeme.io import read_meme
import logomaker

In [2]:
# define a function for reading in data and concatenating #
def read_in_sat_mut (path2satmut, chunksize):
    # define a list for concatenating #
    chunks2cat = []
    # iterate through chunked tsv #
    for chunk in tqdm(pd.read_csv(path2satmut, sep = '\t', chunksize=chunksize)):
        chunks2cat.append(chunk)
    # concatenate chunks #
    cat_df = pd.concat(chunks2cat)
    return cat_df

In [3]:
# define function for reformatting collapsed mpac seqlet calls:
def mpac_collapsed_dels (path2bed,
                    motif_dict):
    # open bed file
    bed = pd.read_csv(path2bed, sep = '\t', header = None)
    filtered_bed = bed.copy()
    # Use column 5 (rep_tf) which has the TF with highest |contribution| for multi-TF intervals
    # Try multiple HOCOMOCO suffixes since they vary (.A, .B, .C, .D)
    def get_tf_family(rep_tf, motif_dict):
        for suffix in ['.A', '.B', '.C', '.D']:
            key = f"{rep_tf}_HUMAN.H11MO.0{suffix}"
            if key in motif_dict:
                return motif_dict[key]
        return None
    # reformat the bed for plotting purposes
    bed2plot = pd.DataFrame({'chrom' : filtered_bed[0],
                             'start' : filtered_bed[1],
                             'end' : filtered_bed[2],
                             'tf' : filtered_bed[5],  # Use pre-computed rep_tf from column 5
                             'tf_family' : [get_tf_family(tf, motif_dict) for tf in filtered_bed[5]],
                             'activity_class' : filtered_bed[6],
                             'enhancer_id' : filtered_bed[7]})
    return bed2plot

In [4]:
# open vierstra clusters for collapsing on families instead of tfs
vierstra_motifs = pd.read_excel('/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/analyses/gnomad_buffering_analysis/motif_annotations.xlsx', sheet_name=[0,1])
# make a dictionary out of the clusterIDs and Names - this will be used to generate the final dictionary with the motif names
idName_dict = dict(zip(vierstra_motifs[0]['Cluster_ID'], vierstra_motifs[0]['Name']))
# open the second page and assign the Names to the individual motifs
vierstra_motifs[1].loc[:,'cluster_name'] = [idName_dict.get(i) for i in vierstra_motifs[1]['Cluster_ID']]
# make a dictionary that pulls the motif name as a key and returns the vierstra family as a value
vierstra_motif_dict = dict(zip(vierstra_motifs[1]['Motif'], vierstra_motifs[1]['cluster_name']))

In [5]:
# reformat MPAC collapsed BED files
path2beds = '/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/processed_data/bed_files'
# k562
k_mpac_collapsed_dels = mpac_collapsed_dels(f'{path2beds}/repTF_k562_dELS_seqlets_01.bed',
                                            vierstra_motif_dict)
# hepg2
h_mpac_collapsed_dels = mpac_collapsed_dels(f'{path2beds}/repTF_hepg2_dELS_seqlets_01.bed',
                                            vierstra_motif_dict)
# sknsh
s_mpac_collapsed_dels = mpac_collapsed_dels(f'{path2beds}/repTF_sknsh_dELS_seqlets_01.bed',
                                            vierstra_motif_dict)

In [6]:
# open data from tian
tian_plotted = pd.read_csv('ukb_window_checklist.with_enhancerid.tsv', sep = '\t')
# open dELS bed file
dELS_bed = pd.read_csv('../../../processed_data/GRCh38-dELS-only.bed', sep = '\t', header = None)
# open full data passed to tian
raw_data = pd.read_csv('openTargets_GWAS_multiTF_seqletFiltered_with_controls.tsv', sep = '\t')

In [7]:
# make some dictionaries for adding data to Tian's df
enhStart = dict(zip(dELS_bed[4], dELS_bed[1]))
# make a dictionary of the rare variants associated with each leadVariant
rareVarsDict = {}
for var in tqdm(tian_plotted['leadVariant'].tolist()):
    # filter the full df for that variant
    rareVars = list(raw_data[(raw_data['leadVariant'] == var) & (raw_data['variant_class'] == 'shadow_emVar')]['openTargets_id'].unique())
    # update dictionary with list of rare variants with that leadVariant
    rareVarsDict[var] = rareVars

100%|██████████| 531/531 [00:00<00:00, 859.33it/s]


In [8]:
# add rare variants to plotted variant df
tian_plotted.loc[:, 'rareVars'] = [rareVarsDict.get(i) for i in tian_plotted['leadVariant']]
# add enhancer start to plotted variant df
tian_plotted.loc[:, 'enhancer_start'] = [enhStart.get(i) for i in tian_plotted['enhancer_id']]
# add filter id
tian_plotted.loc[:, 'filter_id'] = [(':').join([lead, enh, cell, pheno]) for lead, enh, cell, pheno in zip(tian_plotted['leadVariant'], tian_plotted['enhancer_id'], tian_plotted['cell_type'], tian_plotted['Pheno_name'])]

In [9]:
tian_plotted.head()

,pheno_field_id,Pheno_name,trait_type,trait_source,leadVariant,cell_type,n_subjects,n_rare_carriers,n_rare_vids,sd_base,...,z_lead,z_rare,z_rare_minus_lead,is_phenocopy,lead_keep_gts,lead_dropped_gts,enhancer_id,rareVars,enhancer_start,filter_id
0,row_number485_field_id30750,Glycated haemoglobin HbA1c levels (UKB data fi...,continuous,auto-detected,1_13487627_C_T,sknsh,214660,2,1,6.827835,...,0.120159,4.738733,4.618575,True,"Ref/Ref,Ref/Alt",Alt/Alt,EH38E2787464,[1_13487625_A_G],13487532,1_13487627_C_T:EH38E2787464:sknsh:Glycated hae...
1,row_number254_field_id30870,Triglycerides levels (UKB data field 30870),continuous,metadata,18_48073278_C_G,k562,215200,1,1,1.032152,...,0.443014,3.259284,2.816271,True,"Ref/Ref,Ref/Alt",Alt/Alt,EH38E1914587,[18_48073276_C_T],48073234,18_48073278_C_G:EH38E1914587:k562:Triglyceride...
2,row_number128_field_id23108,Impedance of leg left (UKB data field 23108),continuous,metadata,18_48073278_C_G,k562,222968,1,1,35.733592,...,-0.154701,-1.346003,1.191302,True,"Ref/Ref,Ref/Alt",Alt/Alt,EH38E1914587,[18_48073276_C_T],48073234,18_48073278_C_G:EH38E1914587:k562:Impedance of...
3,row_number101_field_id30050,Mean corpuscular haemoglobin (UKB data field 3...,continuous,metadata,18_48073278_C_G,k562,218938,1,1,1.935379,...,-0.008531,-0.949725,0.941194,True,"Ref/Ref,Ref/Alt",Alt/Alt,EH38E1914587,[18_48073276_C_T],48073234,18_48073278_C_G:EH38E1914587:k562:Mean corpusc...
4,row_number473_field_id1239,Current tobacco smoking (UKB data field 1239),continuous,metadata,2_149220012_A_C,k562,226221,4,1,0.414145,...,0.330277,0.884456,0.554179,True,"Ref/Ref,Ref/Alt",Alt/Alt,EH38E3377181,[2_149220009_A_G],149219895,2_149220012_A_C:EH38E3377181:k562:Current toba...


In [10]:
# define a function for returning a dictionary with everything needed for plotting
def dict4plot_v2 (raw_dataDF):
    dict2return = {}
    # add cell types
    dict2return['K562'] = {}
    dict2return['HEPG2'] = {}
    dict2return['SKNSH'] = {}
    # iterate through the unique tissue : variant : gene pairs from the GTEx data I pulled
    for association in tqdm(raw_dataDF['filter_id'].unique()):
        # break out the association id
        leadVar = association.split(':')[0]
        enhancer = association.split(':')[1]
        phenotype = association.split(':')[-1]
        varAll = raw_dataDF[(raw_dataDF['leadVariant'] == leadVar) & (raw_dataDF['Pheno_name'] == phenotype)].copy()
        # check the number of cell types predicted in that cell type
        for cell in varAll['cell_type'].unique():
            # filter that cell type
            cellAll = varAll[varAll['cell_type'] == cell].copy()
            # get rare variants
            rareVariants = cellAll['rareVars'].tolist()[0]
            # combine with gene id to prevent overwrite
            fullID = f'{enhancer}_{phenotype}_{leadVar}'
            # update that cell type's dict
            dict2return[cell.upper()][fullID] = {'chrom' : f'chr{leadVar.split('_')[0]}',
                                                 'leadVar' : leadVar,
                                                 'phenotype' : phenotype,
                                                 'leadVar_pos' : int(leadVar.split('_')[1]),
                                                 'leadVar_z' : cellAll['z_lead'].tolist()[0],
                                                 'rareVars' : rareVariants,
                                                 'rareVars_z' : cellAll['z_rare'].tolist()[0],
                                                 'enhStart' : cellAll['enhancer_start'].tolist()[0],
                                                 'is_phenocopy' : cellAll['is_phenocopy'].tolist()[0]}
                
    return dict2return

In [11]:
dict2plot_v2 = dict4plot_v2(tian_plotted)

100%|██████████| 531/531 [00:00<00:00, 933.51it/s]


In [21]:
# load all tensors
allTensors = {}
for i in tqdm(range(1,23)):
    allTensors[f'chr{i}'] = torch.load(f'/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/distal_cre_sat_mut/encode_cCRE_all/processed_data/seqlets_calling/seqlets_by_chrom/chr{i}_plotLogo_tensors.pt')

100%|██████████| 22/22 [07:45<00:00, 21.16s/it]


In [23]:
def plot_specific_enhancers(cell,
                            annotation_dict,
                            collapsedSeqletBED):
    # define highlight color
    if cell == 'K562':
        highlight = '#00A79D'
    elif cell == 'HepG2':
        highlight = '#FBB040'
    elif cell == 'SKNSH':
        highlight = '#ED1C24'
    # iterate through all of that cell type's candidates
    for candidate in annotation_dict[cell].keys():
        # get the enhancer id
        enhancer_id = candidate.split('_')[0]
        # get the lead variant
        leadVarID = annotation_dict[cell][candidate].get('leadVar')
        # get the chromosome
        chrom = annotation_dict[cell][candidate].get('chrom')
        # get the start position
        enhStart = annotation_dict[cell][candidate].get('enhStart')
        # get the phenotype
        pheno = annotation_dict[cell][candidate].get('phenotype')
        # abridge the phenotype for saving
        phenobridge = pheno.split(' (UKB data field ')[1].split(')')[0]
        # get the lead variant position
        LeadVar = annotation_dict[cell][candidate].get('leadVar_pos')
        # get the z scores
        zLead = float(annotation_dict[cell][candidate].get('leadVar_z'))
        zRare = float(annotation_dict[cell][candidate].get('rareVars_z'))
        # get rare variants
        otherVars = annotation_dict[cell][candidate].get('rareVars')
        # get phenocopy status
        phenocopy = annotation_dict[cell][candidate].get('is_phenocopy')
        if phenocopy == False:
            titleCol = 'red'
        else:
            titleCol = 'green'
        # load the tensors
        plotLogo_tensors = allTensors[chrom]
        
        # get the seqlets for the BED file
        enhBED = collapsedSeqletBED[collapsedSeqletBED['enhancer_id'] == enhancer_id]
        # pull the tensors for the enhancer for that cell type
        if cell == 'HEPG2':
            tensors2plot = plotLogo_tensors['HepG2'][enhancer_id]
        else:
            tensors2plot = plotLogo_tensors[cell][enhancer_id]
    
        # get the max/min attributions for highlighting seqlets
        minAttrib = tensors2plot.min().min() * 1.05
        maxAttrib = tensors2plot.max().max() * 1.05
    
        # build the DF for plotting with logomaker
        dfFromTensor = pd.DataFrame(tensors2plot.numpy()).T.copy()
        renamedDF = dfFromTensor.rename(columns={0: 'A', 1: 'C', 2: 'G', 3: 'T'})
    
        # remove zero-padded positions at ends
        renamedDF = renamedDF.loc[renamedDF.sum(axis=1) != 0]
    
        # adjust enhStart to account for left padding, then reset index to start at 0
        enhStart = enhStart + renamedDF.index[0]
        renamedDF = renamedDF.reset_index(drop=True)
        seq_length = len(renamedDF)

        # set plot parameters
        matplotlib.rcParams['pdf.fonttype'] = 42
        matplotlib.rcParams['ps.fonttype'] = 42
        matplotlib.rcParams['figure.dpi'] = 300
        
        # generate base plot with extra space for annotations
        fig, ax = plt.subplots(figsize=[20, 2.5])
        logoCheck = logomaker.Logo(renamedDF, ax=ax)
        
        # set y-limits based on data only
        logoCheck.ax.set_ylim(minAttrib, maxAttrib)
        
        # track annotation positions to avoid collisions
        top_annotations = []
        bottom_annotations = []
        min_spacing = 3
        # define helper function for plotting
        def get_annotation_level(x_center, text_half_width, existing_annotations):
                    """Find the lowest level where this annotation won't overlap with others."""
                    level = 0
                    while True:
                        conflict = False
                        for other_center, other_half_width, other_level in existing_annotations:
                            if other_level == level:
                                distance = abs(x_center - other_center)
                                min_dist = text_half_width + other_half_width + min_spacing
                                if distance < min_dist:
                                    conflict = True
                                    break
                        if not conflict:
                            return level
                        level += 1

        # highlight seqlets and collect annotation info
        for idx, row in enhBED.iterrows():
            start, stop, actClass, tf = row[['start', 'end', 'activity_class', 'tf_family']].tolist()
            start = int(start - enhStart)
            stop = int(stop - enhStart - 1)
            x_center = (stop + start) / 2
            x_center_norm = x_center / seq_length  # normalize to 0-1 for axes coords
            tf_label = tf.split('_')[0]
            text_half_width = len(tf_label) * 2.5
            
            if actClass == 'Activator':
                logoCheck.highlight_position_range(pmin=start, pmax=stop, alpha=0.5, color='gainsboro', edgecolor='k', floor=0)
                
                level = get_annotation_level(x_center, text_half_width, top_annotations)
                top_annotations.append((x_center, text_half_width, level))
                
                # annotate in axes coords (outside plot area)
                y_offset = 1.05 + level * 0.05
                ax.annotate(tf_label,
                            xy=(x_center_norm, 1.0),  # anchor at top of axes
                            xycoords='axes fraction',
                            xytext=(x_center_norm, y_offset),
                            textcoords='axes fraction',
                            fontsize=8, fontstyle='italic',
                            ha='center', va='bottom',
                            annotation_clip=False)
            else:
                logoCheck.highlight_position_range(pmin=start, pmax=stop, alpha=0.5, color='gainsboro', edgecolor='k', ceiling=0)
                
                level = get_annotation_level(x_center, text_half_width, bottom_annotations)
                bottom_annotations.append((x_center, text_half_width, level))
                
                # annotate in axes coords (outside plot area)
                y_offset = -0.01 - level * 0.12
                ax.annotate(tf_label,
                            xy=(x_center_norm, 0.0),  # anchor at bottom of axes
                            xycoords='axes fraction',
                            xytext=(x_center_norm, y_offset),
                            textcoords='axes fraction',
                            fontsize=8, fontstyle='italic',
                            ha='center', va='top',
                            annotation_clip=False)
        # highlight lead Variant
        LeadVar = int(LeadVar - enhStart - 1)
        lead_contribution = renamedDF.iloc[LeadVar].sum()
        if lead_contribution >= 0:
            logoCheck.highlight_position(p=LeadVar, color='red', alpha=0.5, floor=0)
        else:
            logoCheck.highlight_position(p=LeadVar, color='red', alpha=0.5, ceiling=0)
        # highlight larger effect variants
        for var in otherVars:
            var = int(var.split('_')[1])
            var2plot = int(var - enhStart - 1)
            var_contribution = renamedDF.iloc[var2plot].sum()
            if var_contribution >= 0:
                logoCheck.highlight_position(p=var2plot, color='deepskyblue', alpha=0.5, floor=0)
            else:
                logoCheck.highlight_position(p=var2plot, color='deepskyblue', alpha=0.5, ceiling=0)
        
        logoCheck.style_spines(visible=False)
        # set x-axis ticks to genomic positions at 50bp intervals
        first_tick_genomic = ((enhStart // 50) + 1) * 50
        tick_positions = []
        tick_labels = []
        
        for genomic_pos in range(first_tick_genomic, enhStart + seq_length, 50):
            plot_pos = genomic_pos - enhStart
            if 0 <= plot_pos < seq_length:
                tick_positions.append(plot_pos)
                tick_labels.append(str(genomic_pos))
        
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, rotation=45, ha='right', rotation_mode='anchor')
        ax.tick_params(axis='x', which='major', length=5, pad=5)

        first_minor_genomic = ((enhStart // 10) + 1) * 10
        first_major_genomic = ((enhStart // 50) + 1) * 50

        major_tick_positions = []
        major_tick_labels = []
        minor_tick_positions = []

        for genomic_pos in range(first_minor_genomic, enhStart + seq_length, 10):
            plot_pos = genomic_pos - enhStart - 1
            if 0 <= plot_pos < seq_length:
                if genomic_pos % 50 == 0:  # major tick
                    major_tick_positions.append(plot_pos)
                    major_tick_labels.append(str(genomic_pos))
                else:  # minor tick
                    minor_tick_positions.append(plot_pos)

        ax.set_xticks(major_tick_positions)
        ax.set_xticklabels(major_tick_labels, rotation=45, ha='right', rotation_mode='anchor')
        ax.tick_params(axis='x', which='major', length=5, pad=5)

        ax.set_xticks(minor_tick_positions, minor=True)
        ax.tick_params(axis='x', which='minor', length=2.5)
        
        max_top_level = max([level for _, _, level in top_annotations], default=-1)
        title_y = 1.10 + (max_top_level + 1) * 0.05
        # make title
        ax.set_title(f'{leadVarID} | {pheno} | {enhancer_id} | {cell} | Lead Z: {zLead:.2} | Rare Z: {zRare:.2}', loc='left', y=title_y, color=titleCol, fontweight='bold')
        
        # adjust subplot to make room for annotations
        plt.subplots_adjust(top=0.75, bottom=0.25)
        plt.tight_layout()
        if phenocopy == True:
            plt.savefig(f'plots_out/ukbb/phenocopy_true/{cell}/{enhancer_id}_{phenobridge}_{leadVarID}_{cell}.png', dpi=300)
        else:
            plt.savefig(f'plots_out/ukbb/phenocopy_false/{cell}/{enhancer_id}_{phenobridge}_{leadVarID}_{cell}.png', dpi=300)
        plt.close()

In [24]:
# plot k562 examples
plot_specific_enhancers(
    'K562',
    dict2plot_v2,
    k_mpac_collapsed_dels
)

In [26]:
# plot hepg2 examples
plot_specific_enhancers(
    'HEPG2',
    dict2plot_v2,
    h_mpac_collapsed_dels
)

In [ ]:
# plot sknsh examples
plot_specific_enhancers(
    'SKNSH',
    dict2plot_v2,
    s_mpac_collapsed_dels
)